In [1]:
import pandas as pd

data = pd.read_csv("data_zaid.csv")
data

,Unnamed: 0,IDBeneficiario,IDVivienda,IDPrograma,NombrePrograma,IDEstatusBeneficiario,Descripcion,TipoPobreza,Situacion_Hogar,Ingreso,Descripcion_Inconsistencia
0,0,49417,16565,NaN,NaN,NaN,NaN,Vulnerable por carencia social,No vulnerable por Ingreso,4122.68,NaN
1,1,49419,16566,NaN,NaN,NaN,NaN,Faltan datos,Linea de pobreza extrema por Ingreso,1871.55,Error en caractersiticas de la vivienda
2,2,49418,16566,NaN,NaN,NaN,NaN,Faltan datos,Linea de pobreza extrema por Ingreso,1871.55,"Error en estudios de Jefe de Familia,Error en ..."
3,3,49440,16573,NaN,NaN,NaN,NaN,Pobreza moderada,Linea de pobreza por Ingreso,2505.23,NaN
4,4,49441,16574,13.0,Programa de Inclusión para Personas con Discap...,1.0,Solicitante por Validar,Pobreza moderada,Linea de pobreza por Ingreso,2045.63,NaN
...,...,...,...,...,...,...,...,...,...,...,...
134239,134239,158440,53651,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Error en CURP
134240,134240,158703,53742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
134241,134241,160053,54163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Error CURP Duplicados
134242,134242,157709,53390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
from utils.dict_utils import ESTATUS

data["IDEstatusBeneficiario"].map(ESTATUS).value_counts()

Validando         8969
Beneficiado(a)    5014
Validado           794
Rechazado           77
Cancelada           33
Name: IDEstatusBeneficiario, dtype: int64

In [6]:
data["NombrePrograma"] = data["NombrePrograma"].fillna("Sin programa")
programa_df = data
programa_df = programa_df.drop("Unnamed: 0", axis=1)
programa_df["Status"] = programa_df["IDEstatusBeneficiario"].map(ESTATUS)
programa_df

,IDBeneficiario,IDVivienda,IDPrograma,NombrePrograma,IDEstatusBeneficiario,Descripcion,TipoPobreza,Situacion_Hogar,Ingreso,Descripcion_Inconsistencia,Status
0,49417,16565,NaN,Sin programa,NaN,NaN,Vulnerable por carencia social,No vulnerable por Ingreso,4122.68,NaN,NaN
1,49419,16566,NaN,Sin programa,NaN,NaN,Faltan datos,Linea de pobreza extrema por Ingreso,1871.55,Error en caractersiticas de la vivienda,NaN
2,49418,16566,NaN,Sin programa,NaN,NaN,Faltan datos,Linea de pobreza extrema por Ingreso,1871.55,"Error en estudios de Jefe de Familia,Error en ...",NaN
3,49440,16573,NaN,Sin programa,NaN,NaN,Pobreza moderada,Linea de pobreza por Ingreso,2505.23,NaN,NaN
4,49441,16574,13.0,Programa de Inclusión para Personas con Discap...,1.0,Solicitante por Validar,Pobreza moderada,Linea de pobreza por Ingreso,2045.63,NaN,Validando
...,...,...,...,...,...,...,...,...,...,...,...
134239,158440,53651,NaN,Sin programa,NaN,NaN,NaN,NaN,NaN,Error en CURP,NaN
134240,158703,53742,NaN,Sin programa,NaN,NaN,NaN,NaN,NaN,NaN,NaN
134241,160053,54163,NaN,Sin programa,NaN,NaN,NaN,NaN,NaN,Error CURP Duplicados,NaN
134242,157709,53390,NaN,Sin programa,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
programa_df.loc[~programa_df["Descripcion_Inconsistencia"].isnull(), "Descripcion_Inconsistencia"] = "Datos con inconsistencia"

In [25]:
from codecs import ignore_errors


individuos_dummies = pd.get_dummies(programa_df[["Status", "Descripcion_Inconsistencia"]], prefix="", prefix_sep="")
individuos_dummies[["NombrePrograma", "Status"]] = programa_df[["NombrePrograma", "Status"]]
individuos_df = individuos_dummies.groupby("NombrePrograma").sum()
individuos_df["total"] = individuos_df[list(individuos_df.columns)].sum(axis=1)
individuos_df 

,Beneficiado(a),Cancelada,Rechazado,Validado,Validando,Datos con inconsistencia,total
NombrePrograma,,,,,,,
Hambre Cero,2981.0,8.0,1.0,353.0,1717.0,94.0,5154.0
Impulso a Cuidadoras,1122.0,15.0,2.0,439.0,1433.0,28.0,3039.0
Programa de Inclusión para Personas con Discapacidad en condición de vulnerabilidad,911.0,10.0,74.0,2.0,5819.0,237.0,7053.0
Sin programa,0.0,0.0,0.0,0.0,0.0,25841.0,25841.0


In [22]:
# total_series = individuos_df.sum(numeric_only=True)
# total_series.name = "Total"
# individuos_df = individuos_df.append(total_series)
# individuos_df

C:\Users\zaid_\AppData\Local\Temp\ipykernel_14864\965252130.py:3: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.



,Beneficiado(a),Cancelada,Rechazado,Validado,Validando,Datos con inconsistencia
NombrePrograma,,,,,,
Hambre Cero,2981.0,8.0,1.0,353.0,1717.0,94.0
Impulso a Cuidadoras,1122.0,15.0,2.0,439.0,1433.0,28.0
Programa de Inclusión para Personas con Discapacidad en condición de vulnerabilidad,911.0,10.0,74.0,2.0,5819.0,237.0
Sin programa,0.0,0.0,0.0,0.0,0.0,25841.0
Total,5014.0,33.0,77.0,794.0,8969.0,26200.0


In [27]:
import plotly.express as px

px.pie(individuos_df, names=individuos_df.index, values=individuos_df["total"])

In [10]:
pobreza_dummies = pd.get_dummies(programa_df[["Status", "Descripcion_Inconsistencia"]], prefix="", prefix_sep="")
pobreza_dummies[["NombrePrograma", "Status", "TipoPobreza"]] = programa_df[["NombrePrograma", "Status", "TipoPobreza"]]
pobreza_df = pobreza_dummies.groupby("TipoPobreza").sum()
pobreza_df["total"] = pobreza_df[list(pobreza_df.columns)].sum(axis=1)
pobreza_df

,Beneficiado(a),Cancelada,Rechazado,Validado,Validando,Datos con inconsistencia,total
TipoPobreza,,,,,,,
Faltan datos,26.0,0.0,0.0,1.0,34.0,8492.0,8553.0
No pobre y no Vulnerable,6.0,0.0,8.0,0.0,91.0,671.0,776.0
Pobreza extrema,1781.0,6.0,5.0,181.0,898.0,412.0,3283.0
Pobreza moderada,3070.0,26.0,49.0,608.0,7366.0,3189.0,14308.0
Vulnerable por carencia social,52.0,1.0,11.0,2.0,340.0,896.0,1302.0
Vulnerable por ingreso,57.0,0.0,0.0,2.0,88.0,515.0,662.0


In [11]:
import plotly.express as px

px.pie(pobreza_df, names=pobreza_df.index, values=pobreza_df["total"])

In [12]:
pobreza_df = pobreza_df.drop("Faltan datos", axis=0)

px.pie(pobreza_df, names=pobreza_df.index, values=pobreza_df["total"])

In [13]:
programa_df_por_vivienda = pd.get_dummies(programa_df[["NombrePrograma","Status"]], prefix="", prefix_sep="")
programa_df_por_vivienda[["IDVivienda","TipoPobreza"]] = programa_df[["IDVivienda","TipoPobreza"]]
cols = programa_df_por_vivienda.columns.tolist()
cols = cols[-2:] + cols[:-2]
programa_df_por_vivienda = programa_df_por_vivienda[cols]
programa_df_por_vivienda

,IDVivienda,TipoPobreza,Hambre Cero,Impulso a Cuidadoras,Programa de Inclusión para Personas con Discapacidad en condición de vulnerabilidad,Sin programa,Beneficiado(a),Cancelada,Rechazado,Validado,Validando
0,16565,Vulnerable por carencia social,0,0,0,1,0,0,0,0,0
1,16566,Faltan datos,0,0,0,1,0,0,0,0,0
2,16566,Faltan datos,0,0,0,1,0,0,0,0,0
3,16573,Pobreza moderada,0,0,0,1,0,0,0,0,0
4,16574,Pobreza moderada,0,0,1,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
134239,53651,NaN,0,0,0,1,0,0,0,0,0
134240,53742,NaN,0,0,0,1,0,0,0,0,0
134241,54163,NaN,0,0,0,1,0,0,0,0,0
134242,53390,NaN,0,0,0,1,0,0,0,0,0


In [14]:
vivienda_df = programa_df_por_vivienda.groupby("IDVivienda").sum()
vivienda_df["Habitantes"] = vivienda_df["Hambre Cero"] + vivienda_df["Impulso a Cuidadoras"] + vivienda_df["Programa de Inclusión para Personas con Discapacidad en condición de vulnerabilidad"] + vivienda_df["Sin programa"]
vivienda_df["Pobreza"] = programa_df_por_vivienda["TipoPobreza"]
vivienda_df

,Hambre Cero,Impulso a Cuidadoras,Programa de Inclusión para Personas con Discapacidad en condición de vulnerabilidad,Sin programa,Beneficiado(a),Cancelada,Rechazado,Validado,Validando,Habitantes,Pobreza
IDVivienda,,,,,,,,,,,
6589,0,0,0,2,0,0,0,0,0,2,Pobreza moderada
6590,0,0,0,2,0,0,0,0,0,2,Pobreza extrema
6591,0,0,0,1,0,0,0,0,0,1,Faltan datos
6592,0,1,0,4,1,0,0,0,0,5,Faltan datos
6593,0,0,0,1,0,0,0,0,0,1,Faltan datos
...,...,...,...,...,...,...,...,...,...,...,...
54235,1,0,0,2,0,0,0,0,1,3,Pobreza moderada
54236,1,0,0,2,0,0,0,0,1,3,Pobreza moderada
54237,0,0,0,2,0,0,0,0,0,2,Pobreza moderada


In [15]:
cols = vivienda_df.columns.tolist()
cols = cols[-2:-1] + cols[:4] + cols[4:-2] +cols[-1:]
cols

['Habitantes',
 'Hambre Cero',
 'Impulso a Cuidadoras',
 'Programa de Inclusión para Personas con Discapacidad en condición de vulnerabilidad',
 'Sin programa',
 'Beneficiado(a)',
 'Cancelada',
 'Rechazado',
 'Validado',
 'Validando',
 'Pobreza']

In [16]:
vivienda_df = vivienda_df[cols]
vivienda_df

,Habitantes,Hambre Cero,Impulso a Cuidadoras,Programa de Inclusión para Personas con Discapacidad en condición de vulnerabilidad,Sin programa,Beneficiado(a),Cancelada,Rechazado,Validado,Validando,Pobreza
IDVivienda,,,,,,,,,,,
6589,2,0,0,0,2,0,0,0,0,0,Pobreza moderada
6590,2,0,0,0,2,0,0,0,0,0,Pobreza extrema
6591,1,0,0,0,1,0,0,0,0,0,Faltan datos
6592,5,0,1,0,4,1,0,0,0,0,Faltan datos
6593,1,0,0,0,1,0,0,0,0,0,Faltan datos
...,...,...,...,...,...,...,...,...,...,...,...
54235,3,1,0,0,2,0,0,0,0,1,Pobreza moderada
54236,3,1,0,0,2,0,0,0,0,1,Pobreza moderada
54237,2,0,0,0,2,0,0,0,0,0,Pobreza moderada


In [17]:
total_series = vivienda_df.sum()
total_series.name = "Total"
vivienda_df.append(total_series)

C:\Users\zaid_\AppData\Local\Temp\ipykernel_14864\1337815747.py:1: FutureWarning:

Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.

C:\Users\zaid_\AppData\Local\Temp\ipykernel_14864\1337815747.py:3: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.



,Habitantes,Hambre Cero,Impulso a Cuidadoras,Programa de Inclusión para Personas con Discapacidad en condición de vulnerabilidad,Sin programa,Beneficiado(a),Cancelada,Rechazado,Validado,Validando,Pobreza
IDVivienda,,,,,,,,,,,
6589,2,0,0,0,2,0,0,0,0,0,Pobreza moderada
6590,2,0,0,0,2,0,0,0,0,0,Pobreza extrema
6591,1,0,0,0,1,0,0,0,0,0,Faltan datos
6592,5,0,1,0,4,1,0,0,0,0,Faltan datos
6593,1,0,0,0,1,0,0,0,0,0,Faltan datos
...,...,...,...,...,...,...,...,...,...,...,...
54236,3,1,0,0,2,0,0,0,0,1,Pobreza moderada
54237,2,0,0,0,2,0,0,0,0,0,Pobreza moderada
54238,4,0,0,0,4,0,0,0,0,0,Pobreza moderada


In [31]:
px.histogram(vivienda_df, x="Habitantes", color="Pobreza")

In [18]:
pobreza_vivienda_df = vivienda_df.groupby("Pobreza").sum()
pobreza_vivienda_df["Habitantes"]

Pobreza
Faltan datos                      23459.0
No pobre y no Vulnerable           6087.0
Pobreza extrema                   14161.0
Pobreza moderada                  66031.0
Vulnerable por carencia social    12254.0
Vulnerable por ingreso             7833.0
Name: Habitantes, dtype: float64

In [19]:
pobreza_vivienda_df

,Habitantes,Hambre Cero,Impulso a Cuidadoras,Programa de Inclusión para Personas con Discapacidad en condición de vulnerabilidad,Sin programa,Beneficiado(a),Cancelada,Rechazado,Validado,Validando
Pobreza,,,,,,,,,,
Faltan datos,23459.0,845.0,520.0,1304.0,20790.0,843.0,3.0,15.0,131.0,1677.0
No pobre y no Vulnerable,6087.0,221.0,125.0,296.0,5445.0,210.0,1.0,4.0,43.0,384.0
Pobreza extrema,14161.0,554.0,322.0,710.0,12575.0,552.0,7.0,7.0,93.0,927.0
Pobreza moderada,66031.0,2546.0,1482.0,3323.0,58680.0,2532.0,14.0,39.0,397.0,4369.0
Vulnerable por carencia social,12254.0,439.0,281.0,587.0,10947.0,442.0,5.0,8.0,62.0,790.0
Vulnerable por ingreso,7833.0,291.0,168.0,370.0,7004.0,293.0,2.0,3.0,34.0,497.0


In [20]:
import plotly.express as px

px.pie(pobreza_vivienda_df, names=pobreza_vivienda_df.index, values=pobreza_vivienda_df["Habitantes"])

In [21]:
import plotly.express as px

pobreza_vivienda_df = pobreza_vivienda_df.drop("Faltan datos")

px.pie(pobreza_vivienda_df, names=pobreza_vivienda_df.index, values=pobreza_vivienda_df["Habitantes"])